# 14 - Real Image Binary Inference and Grad-CAM

**Purpose:** Run the binary cancer-risk model on real/external images and generate
Grad-CAM visualisations to qualitatively inspect where the model attends.

**Rules:**
- No training.
- No modifications to original images.
- No dataset manifest changes.
- Binary model only — no multiclass.

> **Medical note:** This output is for demonstration/screening support only and
> is not a diagnosis. Real external images may be out-of-distribution compared
> with the training dataset.

---


## Section 0 - Imports and Environment

In [30]:
import os
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

CUDA_AVAILABLE = torch.cuda.is_available()
try:
    DEVICE   = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "N/A"
except AssertionError:
    DEVICE, GPU_NAME, CUDA_AVAILABLE = torch.device("cpu"), "N/A", False

print(f"torch       : {torch.__version__}")
print(f"CUDA        : {CUDA_AVAILABLE}  GPU: {GPU_NAME}")
print(f"Device      : {DEVICE}")


torch       : 2.6.0+cu124
CUDA        : True  GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Device      : cuda


## Section 1 - Paths and Configuration

In [31]:
OUTPUT_ROOT   = Path(r"C:\SKIN CANCER v2\pipe output")
TRAIN_2PH_DIR = OUTPUT_ROOT / "pytorch_training_binary_2phase"
GRADCAM_DIR   = OUTPUT_ROOT / "real_image_binary_inference_gradcam"
OVERLAY_DIR   = GRADCAM_DIR / "real_image_gradcam_overlays"
MODEL_PATH    = TRAIN_2PH_DIR / "best_model_binary_2phase.pt"

# ── Real image source ─────────────────────────────────────────────────────────
# Set REAL_IMAGE_DIR to your folder of real/external images, OR
# populate REAL_IMAGE_PATHS with explicit file paths.
# If both are empty/missing, the notebook will use a fallback from the test set.
REAL_IMAGE_DIR   = Path(r"C:/SKIN CANCER v2/real_images")   # <- change this
REAL_IMAGE_PATHS = []                                         # <- or list paths here
SUPPORTED_EXTS   = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}

YOUDEN_THRESH      = 0.4219
HIGH_RECALL_THRESH = 0.1700
IMG_SIZE      = 224
RESIZE_TO     = 256
BATCH_SIZE    = 16
NUM_WORKERS   = 0
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DISCLAIMER = (
    "This output is for demonstration/screening support only and is not a diagnosis. "
    "Real external images may be out-of-distribution compared with the training dataset."
)

MODEL_VERSION = "binary_2phase_v1"

print("Config loaded.")
print(f"  REAL_IMAGE_DIR  : {REAL_IMAGE_DIR}")
print(f"  REAL_IMAGE_PATHS: {len(REAL_IMAGE_PATHS)} explicit paths")
print(f"  Youden threshold: {YOUDEN_THRESH}")


Config loaded.
  REAL_IMAGE_DIR  : C:\SKIN CANCER v2\real_images
  REAL_IMAGE_PATHS: 0 explicit paths
  Youden threshold: 0.4219


## Section 2 - Create Output Folders

In [32]:
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {GRADCAM_DIR}")
print(f"Ready: {OVERLAY_DIR}")


Ready: C:\SKIN CANCER v2\pipe output\real_image_binary_inference_gradcam
Ready: C:\SKIN CANCER v2\pipe output\real_image_binary_inference_gradcam\real_image_gradcam_overlays


## Section 3 - Load Model

In [33]:
def build_model():
    model = efficientnet_b0(weights=None)
    in_feat = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_feat, 2),
    )
    return model.to(DEVICE)

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {MODEL_PATH}\n"
        "Run 10_pytorch_binary_two_phase_training.ipynb first."
    )

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model = build_model()
model.load_state_dict(checkpoint["model_state"])
model.eval()

ckpt_epoch = checkpoint.get("global_epoch", checkpoint.get("epoch", "?"))
ckpt_phase = checkpoint.get("phase", "?")
print(f"Model loaded: epoch={ckpt_epoch}  phase={ckpt_phase}")
print(f"Parameters  : {sum(p.numel() for p in model.parameters()):,}")


Model loaded: epoch=17  phase=phase2_finetune
Parameters  : 4,010,110


## Section 4 - Resolve Real Image Paths

In [34]:
import json as _json

def resolve_image_paths():
    """Find real images from REAL_IMAGE_DIR or REAL_IMAGE_PATHS.
    Falls back to test-set samples if neither is available."""
    paths = []

    if REAL_IMAGE_PATHS:
        paths = [Path(p) for p in REAL_IMAGE_PATHS if Path(p).exists()]
        print(f"Using {len(paths)} explicit paths ({len(REAL_IMAGE_PATHS)-len(paths)} not found)")
        return paths, "explicit_list"

    if REAL_IMAGE_DIR.exists():
        paths = sorted([
            p for p in REAL_IMAGE_DIR.iterdir()
            if p.is_file() and p.suffix.lower() in SUPPORTED_EXTS
        ])
        if paths:
            print(f"Found {len(paths)} images in {REAL_IMAGE_DIR}")
            return paths, str(REAL_IMAGE_DIR)
        else:
            print(f"  WARNING: {REAL_IMAGE_DIR} exists but contains no supported image files.")
    else:
        print(f"  WARNING: REAL_IMAGE_DIR not found: {REAL_IMAGE_DIR}")

    # ── Fallback: use a sample from the preprocessed test set ─────────────────
    print("\nFALLBACK: using test set samples for demonstration.")
    print("To use real images, set REAL_IMAGE_DIR or REAL_IMAGE_PATHS above.")
    from pathlib import Path as _P
    test_csv = _P(r"C:\SKIN CANCER v2\pipe output\preprocessing\test_manifest_preprocessed.csv")
    if test_csv.exists():
        test_df = __import__('pandas').read_csv(test_csv, low_memory=False)
        sample  = test_df.sample(min(24, len(test_df)), random_state=42)
        paths   = [_P(p) for p in sample["preprocessed_full_path"].tolist()]
        return paths, "test_set_fallback"
    return [], "none"

real_paths, image_source = resolve_image_paths()
print(f"\nImage source : {image_source}")
print(f"Images to process: {len(real_paths)}")

if not real_paths:
    print("\n" + "="*60)
    print("  No real images found and no test fallback available.")
    print("  Set REAL_IMAGE_DIR = Path(r'your\\folder')")
    print("  or populate REAL_IMAGE_PATHS = ['file1.jpg', ...]")
    print("="*60)


Found 4 images in C:\SKIN CANCER v2\real_images

Image source : C:\SKIN CANCER v2\real_images
Images to process: 4


## Section 5 - Inference Transform and Forward Pass

In [35]:
eval_transform = T.Compose([
    T.Resize(RESIZE_TO),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

raw_transform = T.Compose([
    T.Resize(RESIZE_TO),
    T.CenterCrop(IMG_SIZE),
])  # for display (no normalize)


def get_risk_level(prob):
    if prob < HIGH_RECALL_THRESH:
        return "lower"
    elif prob < YOUDEN_THRESH:
        return "moderate"
    else:
        return "higher"


RISK_WORDING = {
    "lower":    "Model-estimated cancer risk is lower, but this does not rule out disease. "
                "Consult a clinician if the lesion changes, bleeds, hurts, or concerns you.",
    "moderate": "Model-estimated cancer risk is moderate. Consider medical review, especially "
                "if the lesion is new, changing, symptomatic, or clinically concerning.",
    "higher":   "Model-estimated cancer risk is higher. A dermatologist or qualified clinician "
                "should review this lesion.",
}

RISK_COLORS = {
    "lower":    "#b7e4b7",
    "moderate": "#ffe599",
    "higher":   "#f4a0a0",
}


## Section 6 - Grad-CAM Implementation

In [36]:
class GradCAM:
    """
    Grad-CAM for EfficientNetB0.
    Hooks the last feature block (model.features[8]) — the top conv before global pooling.
    """
    def __init__(self, model, target_layer=None):
        self.model       = model
        self.activations = None
        self.gradients   = None

        if target_layer is None:
            target_layer = model.features[8]   # last ConvBnAct block

        self._fwd_hook = target_layer.register_forward_hook(self._save_activation)
        self._bwd_hook = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx=1):
        """
        Returns a Grad-CAM heatmap normalised to [0, 1].
        class_idx=1 = cancer_risk class.
        """
        self.model.zero_grad()
        out   = self.model(input_tensor)
        score = out[0, class_idx]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = (weights * self.activations).sum(dim=1, keepdim=True)
        cam     = F.relu(cam)
        cam     = cam.squeeze().cpu().numpy()

        if cam.ndim == 0:               # edge case: 1×1 feature map
            cam = np.array([[float(cam)]])
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

    def remove_hooks(self):
        self._fwd_hook.remove()
        self._bwd_hook.remove()


def overlay_gradcam(pil_img, cam, alpha=0.45, colormap=plt.cm.jet):
    """Blend a Grad-CAM heatmap onto a PIL image."""
    w, h   = pil_img.size
    cam_r  = Image.fromarray(np.uint8(cam * 255)).resize((w, h), Image.BILINEAR)
    cam_np = np.array(cam_r) / 255.0
    colored = colormap(cam_np)[:, :, :3]
    cam_pil = Image.fromarray(np.uint8(colored * 255))
    return Image.blend(pil_img.convert("RGB"), cam_pil, alpha=alpha)


# Quick functional test
_test_tensor = torch.randn(1, 3, 224, 224).to(DEVICE)
_gc = GradCAM(model)
with torch.enable_grad():
    _cam = _gc.generate(_test_tensor, class_idx=1)
_gc.remove_hooks()
print(f"GradCAM smoke test: cam shape={_cam.shape}  "
      f"range=[{_cam.min():.3f}, {_cam.max():.3f}]")
del _test_tensor, _gc, _cam


GradCAM smoke test: cam shape=(7, 7)  range=[0.000, 1.000]


## Section 7 - Process Real Images

In [37]:
results    = []
failed     = []
gradcam_c  = GradCAM(model)
overlays_saved = 0

if not real_paths:
    print("No images to process — check Section 4 for instructions.")
else:
    print(f"Processing {len(real_paths)} image(s)...")
    print("-" * 60)

    for img_path in real_paths:
        img_path = Path(img_path)
        row = {"image_path": str(img_path), "image_filename": img_path.name}
        try:
            pil_raw  = Image.open(img_path).convert("RGB")
            pil_disp = raw_transform(pil_raw)
            tensor   = eval_transform(pil_raw).unsqueeze(0).to(DEVICE)

            # Inference
            with torch.enable_grad():
                with torch.amp.autocast("cuda", enabled=CUDA_AVAILABLE):
                    out = model(tensor)
                probs = torch.softmax(out, dim=1)[0]
                prob  = float(probs[1].item())
                pred  = int(prob >= YOUDEN_THRESH)

            risk = get_risk_level(prob)

            # Grad-CAM
            cam  = gradcam_c.generate(tensor, class_idx=1)
            overlay_img = overlay_gradcam(pil_disp, cam)
            overlay_name = f"gradcam_{img_path.stem}.png"
            overlay_path = OVERLAY_DIR / overlay_name
            overlay_img.save(overlay_path)
            overlays_saved += 1

            row.update({
                "cancer_risk_probability":               round(prob, 6),
                "cancer_risk_percent":                   round(prob * 100, 2),
                "risk_level":                            f"{risk}_model_estimated_risk",
                "binary_prediction_at_youden_threshold": pred,
                "recommended_action":                    RISK_WORDING[risk],
                "gradcam_overlay_path":                  str(overlay_path),
                "disclaimer":                            DISCLAIMER,
                "model_version":                         MODEL_VERSION,
                "processing_status":                     "ok",
                "source":                                image_source,
                "warning": (
                    "Fallback: test-set image, not a real external image."
                    if image_source == "test_set_fallback" else ""
                ),
            })
            print(f"  {img_path.name:<40} p={prob:.4f}  {risk}")

        except Exception as e:
            row.update({"processing_status": "failed", "error": str(e)})
            failed.append(str(img_path))
            print(f"  FAILED: {img_path.name}: {e}")

        results.append(row)

gradcam_c.remove_hooks()
print(f"\nDone. Processed={len(results)-len(failed)}  Failed={len(failed)}")
print(f"Grad-CAM overlays saved: {overlays_saved}")


Processing 4 image(s)...
------------------------------------------------------------
  ABDO.jpeg                                p=0.0146  lower
  BCC.jpeg                                 p=0.9922  higher
  CANCER.jpeg                              p=0.9946  higher
  MELANOMA.jpeg                            p=0.4441  higher

Done. Processed=4  Failed=0
Grad-CAM overlays saved: 4


## Section 8 - Save Prediction CSV and Summary

In [38]:
if results:
    pred_df   = pd.DataFrame(results)
    pred_path = GRADCAM_DIR / "real_image_predictions_binary_2phase.csv"
    pred_df.to_csv(pred_path, index=False)
    print(f"Saved {pred_path.name}  ({len(pred_df)} rows)")

    ok_df = pred_df[pred_df["processing_status"] == "ok"]
    risk_counts = ok_df["risk_level"].value_counts().to_dict() if len(ok_df) else {}

    summary_df = pd.DataFrame([{
        "image_source":       image_source,
        "total_images":       len(results),
        "processed_ok":       len(ok_df),
        "failed":             len(failed),
        "gradcam_overlays":   overlays_saved,
        "lower_risk_count":   risk_counts.get("lower_model_estimated_risk", 0),
        "moderate_risk_count": risk_counts.get("moderate_model_estimated_risk", 0),
        "higher_risk_count":  risk_counts.get("higher_model_estimated_risk", 0),
        "model_checkpoint":   MODEL_PATH.name,
        "youden_threshold":   YOUDEN_THRESH,
        "training_performed": False,
        "images_modified":    False,
    }])
    sum_path = GRADCAM_DIR / "real_image_inference_summary.csv"
    summary_df.to_csv(sum_path, index=False)
    print(f"Saved {sum_path.name}")
else:
    pred_df    = pd.DataFrame()
    summary_df = pd.DataFrame()
    print("No results to save.")


Saved real_image_predictions_binary_2phase.csv  (4 rows)
Saved real_image_inference_summary.csv


## Section 9 - Prediction Grid

In [39]:
if len(pred_df) > 0 and "processing_status" in pred_df.columns:
    ok_df = pred_df[pred_df["processing_status"] == "ok"].reset_index(drop=True)
    n = min(len(ok_df), 24)
    if n > 0:
        N_COLS = 6
        n_rows = (n + N_COLS - 1) // N_COLS
        fig, axes = plt.subplots(n_rows, N_COLS,
                                  figsize=(N_COLS * 2.5, n_rows * 3.0))
        axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
        for ax in axes_flat:
            ax.axis("off")

        for i in range(n):
            ax   = axes_flat[i]
            row  = ok_df.iloc[i]
            prob = float(row["cancer_risk_probability"])
            risk = row["risk_level"].split("_")[0]
            try:
                img = Image.open(row["image_path"]).convert("RGB")
                img = raw_transform(img)
                ax.imshow(img)
            except Exception:
                ax.set_facecolor("#ddd")
            color = RISK_COLORS.get(risk, "white")
            src_note = " [test]" if row.get("source") == "test_set_fallback" else ""
            ax.set_title(
                f"{row['image_filename'][:18]}{src_note}\n"
                f"p={prob:.3f}  {risk.capitalize()}",
                fontsize=6.5, pad=2, backgroundcolor=color,
            )
            ax.axis("off")

        patches = [mpatches.Patch(color=v, label=k.capitalize())
                   for k, v in RISK_COLORS.items()]
        fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=9,
                   title="Risk Level", bbox_to_anchor=(0.5, -0.02))
        plt.suptitle(
            f"Real Image Predictions — Binary Cancer-Risk Model\n"
            f"Source: {image_source}  |  Threshold: {YOUDEN_THRESH} (Youden J)",
            fontsize=11,
        )
        plt.tight_layout()
        grid_path = GRADCAM_DIR / "real_image_prediction_grid.png"
        plt.savefig(grid_path, dpi=100, bbox_inches="tight")
        plt.close()
        print(f"Saved {grid_path.name}")
    else:
        print("No successfully processed images for prediction grid.")
else:
    print("No predictions available for prediction grid.")


Saved real_image_prediction_grid.png


## Section 10 - Grad-CAM Overlay Grid

In [40]:
if len(pred_df) > 0 and "gradcam_overlay_path" in pred_df.columns:
    gc_df = pred_df[pred_df["gradcam_overlay_path"].notna() &
                    (pred_df["processing_status"] == "ok")].reset_index(drop=True)
    n = min(len(gc_df), 24)
    if n > 0:
        N_COLS = 6
        n_rows = (n + N_COLS - 1) // N_COLS
        fig, axes = plt.subplots(n_rows, N_COLS,
                                  figsize=(N_COLS * 2.5, n_rows * 3.0))
        axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
        for ax in axes_flat:
            ax.axis("off")

        for i in range(n):
            ax  = axes_flat[i]
            row = gc_df.iloc[i]
            prob = float(row["cancer_risk_probability"])
            risk = row["risk_level"].split("_")[0]
            try:
                overlay = Image.open(row["gradcam_overlay_path"]).convert("RGB")
                ax.imshow(overlay)
            except Exception:
                ax.set_facecolor("#ddd")
            color = RISK_COLORS.get(risk, "white")
            ax.set_title(
                f"{row['image_filename'][:18]}\n"
                f"p={prob:.3f}  {risk.capitalize()}",
                fontsize=6.5, pad=2, backgroundcolor=color,
            )
            ax.axis("off")

        plt.suptitle(
            "Grad-CAM Overlays — EfficientNetB0 (cancer_risk class)\n"
            "Heat = regions most influential for the cancer-risk prediction",
            fontsize=11,
        )
        plt.tight_layout()
        gc_grid_path = GRADCAM_DIR / "real_image_gradcam_grid.png"
        plt.savefig(gc_grid_path, dpi=100, bbox_inches="tight")
        plt.close()
        print(f"Saved {gc_grid_path.name}")
    else:
        print("No Grad-CAM overlays available for grid.")
else:
    print("No Grad-CAM paths available.")


Saved real_image_gradcam_grid.png


## Section 11 - Save Inference Config

In [41]:
import json as _json
inf_config = {
    "model_version":    MODEL_VERSION,
    "checkpoint":       MODEL_PATH.name,
    "checkpoint_epoch": ckpt_epoch,
    "checkpoint_phase": ckpt_phase,
    "image_source":     image_source,
    "total_images":     len(results),
    "processed_ok":     len([r for r in results if r.get("processing_status")=="ok"]),
    "failed":           failed,
    "youden_threshold": YOUDEN_THRESH,
    "high_recall_threshold": HIGH_RECALL_THRESH,
    "risk_levels": {
        "lower":    f"probability < {HIGH_RECALL_THRESH}",
        "moderate": f"{HIGH_RECALL_THRESH} <= probability < {YOUDEN_THRESH}",
        "higher":   f"probability >= {YOUDEN_THRESH}",
    },
    "gradcam_target_layer": "model.features[8] (last ConvBnAct block)",
    "transform": [f"Resize({RESIZE_TO})", f"CenterCrop({IMG_SIZE})",
                  "ToTensor()", "Normalize(ImageNet)"],
    "training_performed": False,
    "images_modified":    False,
    "disclaimer":         DISCLAIMER,
}
cfg_path = GRADCAM_DIR / "real_image_inference_config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    _json.dump(inf_config, f, indent=2)
print(f"Saved {cfg_path.name}")


Saved real_image_inference_config.json


## Section 12 - Output File Verification

In [42]:
required_files = [
    GRADCAM_DIR / "real_image_predictions_binary_2phase.csv",
    GRADCAM_DIR / "real_image_inference_summary.csv",
    GRADCAM_DIR / "real_image_prediction_grid.png",
    GRADCAM_DIR / "real_image_gradcam_grid.png",
    OVERLAY_DIR,
    GRADCAM_DIR / "real_image_inference_config.json",
]
print("Output file verification:")
all_ok = True
for p in required_files:
    p = Path(p)
    if p.is_dir():
        n_files = len(list(p.iterdir()))
        status  = "OK" if n_files > 0 else "EMPTY"
        print(f"  [{status}] {p.name}/ ({n_files} files)")
    else:
        exists = p.exists()
        size   = p.stat().st_size if exists else 0
        status = "OK" if exists else "MISSING"
        print(f"  [{status}] {p.name:<52} {size:>10,} bytes")
        if not exists: all_ok = False
print("\nAll files present." if all_ok else "\nWARNING: some files missing.")


Output file verification:
  [OK] real_image_predictions_binary_2phase.csv                  2,385 bytes
  [OK] real_image_inference_summary.csv                            280 bytes
  [OK] real_image_prediction_grid.png                          402,630 bytes
  [OK] real_image_gradcam_grid.png                             352,226 bytes
  [OK] real_image_gradcam_overlays/ (4 files)
  [OK] real_image_inference_config.json                            920 bytes

All files present.


## Section 13 - Final Summary (Copy-Paste Ready)

In [43]:
ok_results = [r for r in results if r.get("processing_status") == "ok"]
risk_dist  = {}
for r in ok_results:
    rl = r.get("risk_level", "unknown")
    risk_dist[rl] = risk_dist.get(rl, 0) + 1

print("=" * 70)
print("  14_real_image_binary_inference_gradcam -- FINAL SUMMARY")
print("=" * 70)
print(f"\n 1. Model checkpoint          : {MODEL_PATH.name}")
print(f"    Checkpoint epoch          : {ckpt_epoch}  phase: {ckpt_phase}")
print(f" 2. Real image source          : {image_source}")
print(f" 3. Images processed (ok)      : {len(ok_results)}")
print(f" 4. Failed images              : {len(failed)}")
print(f"\n 5. Thresholds:")
print(f"      Youden J (main)          : {YOUDEN_THRESH}")
print(f"      High-recall (caution)    : {HIGH_RECALL_THRESH}")
print(f"\n 6. Risk level distribution:")
for rl, cnt in sorted(risk_dist.items()):
    print(f"      {rl:<40}: {cnt}")
print(f"\n 7. Output file verification:")
for p in required_files:
    p = Path(p)
    if p.is_dir():
        n_files = len(list(p.iterdir()))
        print(f"      [{'OK' if n_files>0 else 'EMPTY'}] {p.name}/ ({n_files} files)")
    else:
        exists = p.exists()
        size   = p.stat().st_size if exists else 0
        print(f"      [{'OK' if exists else 'MISSING'}] {p.name:<52} {size:>8,} bytes")
print(f"\n 8. Grad-CAM overlays saved    : {overlays_saved}")
print(f" 9. Training performed         : False")
print(f"10. Original images modified   : False")
print(f"\n11. Medical caution:")
print(f"    {DISCLAIMER}")
print("=" * 70)


  14_real_image_binary_inference_gradcam -- FINAL SUMMARY

 1. Model checkpoint          : best_model_binary_2phase.pt
    Checkpoint epoch          : 17  phase: phase2_finetune
 2. Real image source          : C:\SKIN CANCER v2\real_images
 3. Images processed (ok)      : 4
 4. Failed images              : 0

 5. Thresholds:
      Youden J (main)          : 0.4219
      High-recall (caution)    : 0.17

 6. Risk level distribution:
      higher_model_estimated_risk             : 3
      lower_model_estimated_risk              : 1

 7. Output file verification:
      [OK] real_image_predictions_binary_2phase.csv                2,385 bytes
      [OK] real_image_inference_summary.csv                          280 bytes
      [OK] real_image_prediction_grid.png                        402,630 bytes
      [OK] real_image_gradcam_grid.png                           352,226 bytes
      [OK] real_image_gradcam_overlays/ (4 files)
      [OK] real_image_inference_config.json                        

## Section 14 - Completion Summary

**14_real_image_binary_inference_gradcam is complete.**

**What was accomplished:**
- Real/external images loaded from `REAL_IMAGE_DIR` (or fallback to test-set samples).
- Binary inference run on each image using `best_model_binary_2phase.pt`.
- Risk level and recommended wording assigned per image.
- Grad-CAM heatmaps generated using the last EfficientNetB0 feature block.
- Prediction grid and Grad-CAM overlay grid saved.
- All predictions and config saved.

**Grad-CAM note:** Heatmaps are qualitative only. They indicate regions most influential
for the cancer-risk prediction but do not prove the model reasons clinically.

**To use with real images:**
Set `REAL_IMAGE_DIR = Path(r"your_folder")` in Section 1 and re-run.
